Xg boost Model 



In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('Student Social Media And Mental Health Impact.csv')
df.head()

,Age,Gender,Country,Academic_Level,Most_Used_Platform,Purpose_Of_Use,Avg_Daily_Usage_Hours,Daily_Unlocks,Study_Hours,Physical_Activity_Hours,Sleep_Hours_Per_Night,Stress_Level,Mental_Health_Score
0,21,Male,Other,Undergraduate,Facebook,Networking,4.0,134,4.5,2.2,6.7,Medium,6.8
1,23,Female,Other,Graduate,LinkedIn,Education,1.6,73,7.0,2.4,8.6,Low,7.6
2,22,Male,Canada,Undergraduate,Instagram,Entertainment,4.6,166,4.0,1.8,6.7,Medium,7.0
3,18,Male,Other,High School,Snapchat,Entertainment,7.0,220,1.0,1.7,5.4,Very High,5.3
4,24,Female,Other,Graduate,Facebook,Networking,7.5,237,1.0,1.1,5.0,Very High,4.4


### 2. Data Cleaning & Feature Engineering

In [ ]:
#drop duplicates and clip unrealistic physical activity hrs

df = df.drop_duplicates()
df['Physical_Activity_Hours'] = df['Physical_Activity_Hours'].clip(lower=0) # -ve hrs value to 0

def group_countries = df['Country'].value_counts().index[:10]


### 3. Feature Selection & Train-Test Split

In [7]:
skwewd_col         = ['Study_Hours']
other_numeric_cols = ["Age", "Avg_Daily_Usage_Hours", "Daily_Unlocks",
                      "Physical_Activity_Hours", "Sleep_Hours_Per_Night"]
ordinal_col        = ['Stress_Level']
normal_col         = ["Gender", "Academic_Level", "Most_Used_Platform", "Purpose_Of_Use", "Grouped_country"]

feature_col = skwewd_col + other_numeric_cols + ordinal_col + normal_col

X = df[feature_col]
y = df['Mental_Health_Score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

KeyError: "['Grouped_country'] not in index"

### 4. Build Preprocessing Pipelines

In [ ]:
skew_pipeline = Pipeline(steps=[
    ('log_transform', FunctionTransformer(np.log1p)),
    ('scale', StandardScaler())
])

plain_numeric_pipeline = Pipeline(steps=[
    ('scale', StandardScaler())
])

ordinal_pipeline = Pipeline(steps=[
    ('encode', OrdinalEncoder(categories=[['Low', 'Medium', 'High', 'Very High']]))
])

nominal_pipeline = Pipeline(steps=[
    ('encode', OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("Skewed_Pipeline", skew_pipeline, skwewd_col),
    ("Plain_Numeric", plain_numeric_pipeline, other_numeric_cols),
    ('Ordinal', ordinal_pipeline, ordinal_col),
    ('Normal', nominal_pipeline, normal_col)
])

### 5. XGBoost Regressor (Baseline Model)

In [ ]:
xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('xgboost', XGBRegressor(n_estimators=200, learning_rate=0.05, random_state=42))
])

xgb_pipeline.fit(X_train, y_train)

xgb_preds       = xgb_pipeline.predict(X_test)
xgb_preds_train = xgb_pipeline.predict(X_train)

xgb_r2_testing  = r2_score(y_test, xgb_preds)
xgb_r2_training = r2_score(y_train, xgb_preds_train)
xgb_mae         = mean_absolute_error(y_test, xgb_preds)
xgb_mse         = mean_squared_error(y_test, xgb_preds)
xgb_rmse        = np.sqrt(xgb_mse)

print(f"Baseline XGBoost Training R2: {xgb_r2_training:.4f}")
print(f"Baseline XGBoost Testing R2 : {xgb_r2_testing:.4f}")
print(f"Baseline XGBoost MAE        : {xgb_mae:.4f}")
print(f"Baseline XGBoost MSE        : {xgb_mse:.4f}")
print(f"Baseline XGBoost RMSE       : {xgb_rmse:.4f}")

### 6. Hyperparameter Tuning with RandomizedSearchCV

In [ ]:
param_grid = {
    'xgboost__n_estimators': [100, 200, 300, 500],
    'xgboost__learning_rate': [0.01, 0.03, 0.05, 0.1, 0.2],
    'xgboost__max_depth': [3, 4, 6, 8],
    'xgboost__subsample': [0.6, 0.8, 1.0],
    'xgboost__colsample_bytree': [0.6, 0.8, 1.0],
    'xgboost__reg_alpha': [0, 0.1, 1.0],
    'xgboost__reg_lambda': [0.5, 1.0, 2.0]
}

random_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_grid,
    n_iter=15,
    cv=5,
    scoring='r2',
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

### 7. Best Parameters

In [ ]:
print("Best Hyperparameters for XGBoost:")
print(random_search.best_params_)

### 8. Evaluate Tuned XGBoost Model

In [ ]:
xgb_best_pipeline = random_search.best_estimator_

xgb_tuned_preds          = xgb_best_pipeline.predict(X_test)
xgb_tuned_preds_training = xgb_best_pipeline.predict(X_train)

xgb_tuned_r2_test  = r2_score(y_test, xgb_tuned_preds)
xgb_tuned_r2_train = r2_score(y_train, xgb_tuned_preds_training)
xgb_tuned_mae      = mean_absolute_error(y_test, xgb_tuned_preds)
xgb_tuned_mse      = mean_squared_error(y_test, xgb_tuned_preds)
xgb_tuned_rmse     = np.sqrt(xgb_tuned_mse)

print(f"Tuned XGBoost Training R2: {xgb_tuned_r2_train:.4f}")
print(f"Tuned XGBoost Testing R2 : {xgb_tuned_r2_test:.4f}")
print(f"Tuned XGBoost MAE        : {xgb_tuned_mae:.4f}")
print(f"Tuned XGBoost MSE        : {xgb_tuned_mse:.4f}")
print(f"Tuned XGBoost RMSE       : {xgb_tuned_rmse:.4f}")

### 9. Comparison: Baseline vs Tuned XGBoost

In [ ]:
results = pd.DataFrame({
    'Model': ['XGBoost (Baseline)', 'XGBoost (Tuned)'],
    'Training R2': [xgb_r2_training, xgb_tuned_r2_train],
    'Testing R2': [xgb_r2_testing, xgb_tuned_r2_test],
    'MAE': [xgb_mae, xgb_tuned_mae],
    'RMSE': [xgb_rmse, xgb_tuned_rmse]
})

results

### 10. Feature Importance Visualization

In [ ]:
# Extract transformed feature names
feature_names = xgb_best_pipeline.named_steps['preprocessor'].get_feature_names_out()
importances   = xgb_best_pipeline.named_steps['xgboost'].feature_importances_

feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(x=feat_imp.values, y=feat_imp.index)
plt.title('Top 15 Feature Importances (Tuned XGBoost)')
plt.xlabel('Relative Importance')
plt.ylabel('Features')
plt.tight_layout()
plt.show()